# 11 Interpretability and market regimes

SHAP values for the random forest and gradient boosting, and calm against stress months (Chapter 5). The tree models of notebook 08 were not saved: they are refitted year by year with the selected hyperparameters (`trees_checkpoints/*_tuning.json`), the same data and the same seed, and each year the refitted forecasts are checked against those saved by notebook 08. SHAP values are exact (TreeSHAP in LightGBM, `pred_contrib=True`); country and sector indicators are summed into two features. Tables in `tables_ch5/`, figures in `figures_ch5/`; progress is saved year by year in `shap_checkpoints/`. About 25 minutes, no GPU needed.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, re, json, glob, time, gc, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR  = "/content/drive/MyDrive/Thesis/data"
TREE_CKPT = os.path.join(DATA_DIR, "trees_checkpoints")     # notebook 08
SHAP_CKPT = os.path.join(DATA_DIR, "shap_checkpoints")      # this notebook
TAB_DIR   = os.path.join(DATA_DIR, "tables_ch5")
FIG_DIR   = os.path.join(DATA_DIR, "figures_ch5")
for d in (SHAP_CKPT, TAB_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

SAMPLES = {"complete_case_mv25": "Complete case", "imputed_mv25": "Imputed"}
FIRST_TEST, LAST_TEST = 2010, 2025
N_TRAIN, N_VAL = 8, 2
SEED = 0
RF_TREES = 300
GB_MAX_TREES, GB_MIN_TREES, GB_PATIENCE = 1000, 10, 200
TREES = ["rf", "gbrt"]

CHARS = ["size", "bm", "operatingprofitability", "totalassetgrowth", "ret_2_12",
         "issues_1_36", "accrualschange", "roa", "investment", "ret_1",
         "ret_12_36", "dy", "beta", "vola", "turnover", "debttoprice",
         "salestoprice", "cftoprice", "earningstoprice", "issues_1_12",
         "fcdispersion", "grossprofitability"]
FEATURES = CHARS + ["country", "sector"]      # indicators grouped

## Functions

The first four are copied unchanged from notebook 08, so data and windows are identical; the others refit a single model with the selected hyperparameters.

In [ ]:
# Copied unchanged from notebook 08

def load_sample(name, est_dir, panel_dir):
    est = pd.read_parquet(os.path.join(est_dir, f"estimation_{name}.parquet"))
    est["date"] = pd.to_datetime(est["date"])
    raw = pd.read_parquet(os.path.join(panel_dir, "panel_characteristics.parquet"),
                          columns=["symbol", "date", "target"])
    raw["date"] = pd.to_datetime(raw["date"])
    return (est.merge(raw.rename(columns={"target": "raw_target"}), on=["symbol", "date"],
                      how="left").sort_values(["date", "symbol"]).reset_index(drop=True))


def build_parsimonious(df, chars=CHARS):
    dummies = pd.concat([
        pd.get_dummies(df["country"], prefix="cty", drop_first=True, dtype=np.float32),
        pd.get_dummies(df["sector"], prefix="sec", drop_first=True, dtype=np.float32)], axis=1)
    X = np.hstack([df[chars].to_numpy(dtype=np.float32), dummies.to_numpy()])
    return X, list(chars) + list(dummies.columns)


def rolling_splits(dates, first_test_year, last_test_year, n_train=8, n_val=2):
    years = pd.DatetimeIndex(dates).year
    for y in range(first_test_year, last_test_year + 1):
        tr = (years >= y - n_val - n_train) & (years <= y - n_val - 1)
        va = (years >= y - n_val) & (years <= y - 1)
        te = years == y
        if te.sum() and tr.sum() and va.sum():
            yield y, np.asarray(tr), np.asarray(va), np.asarray(te)


def base_params(seed, threads):
    return {"objective": "regression", "verbose": -1, "seed": seed, "deterministic": True,
            "force_row_wise": True, "num_threads": threads, "min_data_in_leaf": 20}


# Refitting and SHAP

def refit_rf(dtr, n_features, depth, mtry, seed=SEED, threads=0):
    # Same parameters as fit_rf in notebook 08, for the selected depth and mtry only
    p = {**base_params(seed, threads), "boosting": "rf", "num_leaves": 2 ** depth,
         "max_depth": depth, "bagging_freq": 1, "bagging_fraction": 0.632,
         "feature_fraction_bynode": min(1.0, mtry / n_features)}
    return lgb.train(p, dtr, num_boost_round=RF_TREES)


def refit_gbrt(dtr, dva, depth, lr, seed=SEED, threads=0):
    # Same procedure as fit_gbrt in notebook 08, for the selected depth and learning rate
    # only, including early stopping on the validation sample: with early stopping
    # LightGBM keeps the trees up to its best iteration, and the forecasts of notebook 08
    # come from that model.
    p = {**base_params(seed, threads), "boosting": "gbdt", "learning_rate": lr,
         "num_leaves": 2 ** depth, "max_depth": depth}
    return lgb.train(p, dtr, num_boost_round=GB_MAX_TREES, valid_sets=[dva], valid_names=["val"],
                     callbacks=[lgb.early_stopping(GB_PATIENCE, first_metric_only=True, verbose=False)])


def tree_shap(model, X, is_rf, num_iteration=None):
    # Exact SHAP values (TreeSHAP) from LightGBM: one column per predictor plus the
    # expected value in the last column. In random-forest mode LightGBM sums the
    # contributions of the trees while the forecast is their average, so the
    # contributions are divided by the number of trees.
    c = model.predict(X, pred_contrib=True, num_iteration=num_iteration)
    if is_rf:
        c = c / model.num_trees()
    return c


def group_shap(c, names):
    # Sum the SHAP values of all country and all sector indicators (additivity)
    names = list(names)
    out = {n: c[:, i] for i, n in enumerate(names) if not n.startswith(("cty_", "sec_"))}
    out["country"] = c[:, [i for i, n in enumerate(names) if n.startswith("cty_")]].sum(axis=1)
    out["sector"] = c[:, [i for i, n in enumerate(names) if n.startswith("sec_")]].sum(axis=1)
    out["bias"] = c[:, -1]
    return pd.DataFrame(out)[FEATURES + ["bias"]].astype(np.float32)

## Refitting and SHAP values

For each test year: refit rf and gbrt with the saved hyperparameters, check the forecasts against those of notebook 08, and compute SHAP values on the test observations.

In [ ]:
t0 = time.time()
for tag, label in SAMPLES.items():
    ck_in, ck_out = os.path.join(TREE_CKPT, tag), os.path.join(SHAP_CKPT, tag)
    os.makedirs(ck_out, exist_ok=True)
    est = load_sample(tag, DATA_DIR, DATA_DIR)
    X, names = build_parsimonious(est)
    y = est["target"].to_numpy()
    with open(os.path.join(ck_in, "names.json")) as fh:
        saved_names = json.load(fh)
    assert names == saved_names, "predictors differ from notebook 08"
    print(f"\n{label}: {len(est):,} security-months, {X.shape[1]} predictors (same as notebook 08)")

    for yr, tr, va, te in rolling_splits(est["date"], FIRST_TEST, LAST_TEST, N_TRAIN, N_VAL):
        f_out = os.path.join(ck_out, f"{yr}_shap.parquet")
        if os.path.exists(f_out):
            print(f"   {yr}: already done, skipped")
            continue
        t1 = time.time()
        with open(os.path.join(ck_in, f"{yr}_tuning.json")) as fh:
            tun = json.load(fh)
        dtr = lgb.Dataset(X[tr], y[tr], params={"verbose": -1}, free_raw_data=False).construct()
        dva = lgb.Dataset(X[va], y[va], reference=dtr).construct()
        rf = refit_rf(dtr, X.shape[1], tun["rf_depth"], tun["rf_mtry"])
        gb = refit_gbrt(dtr, dva, tun["gbrt_depth"], tun["gbrt_lr"])

        Xte = X[te]
        pred = {"rf": rf.predict(Xte), "gbrt": gb.predict(Xte, num_iteration=tun["gbrt_trees"])}
        shap = {"rf": tree_shap(rf, Xte, True),
                "gbrt": tree_shap(gb, Xte, False, num_iteration=tun["gbrt_trees"])}

        # Check 1: refitted forecasts against the forecasts saved by notebook 08
        keys = est.loc[te, ["symbol", "date"]].reset_index(drop=True)
        saved = keys.merge(pd.read_parquet(os.path.join(ck_in, f"{yr}_pred.parquet"))
                           [["symbol", "date", "rf", "gbrt"]], on=["symbol", "date"], how="left")
        check = {"year": yr, "rows": int(te.sum()),
                 "gbrt_trees_recorded": int(tun["gbrt_trees"]),
                 "gbrt_trees_used": int(min(tun["gbrt_trees"], gb.num_trees()))}
        for m in TREES:
            check[f"{m}_max_diff"] = float(np.nanmax(np.abs(pred[m] - saved[m].to_numpy())))
            check[f"{m}_corr"] = float(np.corrcoef(pred[m], saved[m].to_numpy())[0, 1])
            # Check 2: SHAP values add up to the forecast
            check[f"{m}_additivity"] = float(np.max(np.abs(shap[m].sum(axis=1) - pred[m])))

        out = pd.concat([keys, est.loc[te, ["MV"] + CHARS].reset_index(drop=True).astype(
            {c: np.float32 for c in CHARS})], axis=1)
        for m in TREES:
            g = group_shap(shap[m], names)
            g.columns = [f"{m}__{c}" for c in g.columns]
            out = pd.concat([out, g], axis=1)
        with open(os.path.join(ck_out, f"{yr}_check.json"), "w") as fh:
            json.dump(check, fh)
        out.to_parquet(f_out, index=False)            # written last: marks the year complete
        print(f"   {yr}: max |diff| rf {check['rf_max_diff']:.1e}, gbrt {check['gbrt_max_diff']:.1e} | "
              f"gbrt trees {check['gbrt_trees_used']} (recorded {check['gbrt_trees_recorded']}) | "
              f"{time.time() - t1:.0f}s", flush=True)
    del est, X, y; gc.collect()
print(f"\nDone in {(time.time() - t0) / 60:.1f} min.")

### Refitting check

`max_diff`: largest absolute difference from the forecasts of notebook 08. `additivity`: largest gap between the sum of the SHAP values and the forecast. The last two columns compare the gbrt trees recorded in notebook 08 with those used.

In [ ]:
checks = {SAMPLES[t]: pd.DataFrame([json.load(open(f)) for f in
                                    sorted(glob.glob(os.path.join(SHAP_CKPT, t, "*_check.json")))]).set_index("year")
          for t in SAMPLES}
cols = ["rf_max_diff", "gbrt_max_diff", "rf_corr", "gbrt_corr", "rf_additivity", "gbrt_additivity",
        "gbrt_trees_recorded", "gbrt_trees_used"]
for s, c in checks.items():
    print(f"\n{s}:")
    print(c[cols].to_string(float_format=lambda v: f"{v:.2e}" if abs(v) < 1e-2 else f"{v:.4f}"))

## Loading the SHAP values

In [ ]:
S = {}
for tag, label in SAMPLES.items():
    S[label] = pd.concat([pd.read_parquet(f) for f in
                          sorted(glob.glob(os.path.join(SHAP_CKPT, tag, "*_shap.parquet")))],
                         ignore_index=True)
    S[label]["date"] = pd.to_datetime(S[label]["date"])
    print(f"{label:14s} {len(S[label]):>9,} test observations with SHAP values")


def shap_cols(model):
    return [f"{model}__{f}" for f in FEATURES]


def importance(df, model):
    # Mean absolute SHAP value of each characteristic, in percentage points of monthly return
    a = df[shap_cols(model)].astype("float64").abs().mean()
    a.index = FEATURES
    return 100 * a


def shares(imp):
    return 100 * imp / imp.sum()

### LaTeX tables (as in notebook 10)

In [ ]:
LABELS = {"rf", "gbrt", "ols3", "ols_pars", "elanet", "nn_3", "comb"}


def _cell(v, digits):
    if pd.isna(v):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{v:,}"
    s = f"{v:.{digits}f}"
    return f"${s}$" if v < 0 else s


def _tex(s):
    s = str(s)
    if s == "t":
        return "$t$"
    if s in LABELS or s in CHARS:
        return r"\texttt{" + s.replace("_", r"\_") + "}"
    s = re.sub(r"(\d{4})-(\d{4})", r"\1--\2", s)
    s = s.replace("_", r"\_").replace("%", r"\%")
    return re.sub(r"\bR2\b", r"$R^2$", s)


def latex_table(df, fname, caption, label, digits=2, notes=None, size=r"\small"):
    two_level = isinstance(df.columns, pd.MultiIndex)
    ncol = df.shape[1]
    lines = [r"\begin{table}[htbp]", r"\centering", size,
             rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{@{}l" + "r" * ncol + r"@{}}", r"\toprule"]
    if two_level:
        top, bottom = df.columns.get_level_values(0), df.columns.get_level_values(1)
        groups, start = [], 0
        for i in range(1, ncol + 1):
            if i == ncol or top[i] != top[start]:
                groups.append((top[start], start, i))
                start = i
        lines.append(" & " + " & ".join(rf"\multicolumn{{{e - s}}}{{c}}{{{_tex(g)}}}"
                                        for g, s, e in groups) + r" \\")
        lines.append("".join(rf"\cmidrule(lr){{{s + 2}-{e + 1}}}" for g, s, e in groups))
        lines.append(" & " + " & ".join(_tex(b) for b in bottom) + r" \\")
    else:
        lines.append(" & " + " & ".join(_tex(c) for c in df.columns) + r" \\")
    lines.append(r"\midrule")
    for idx, row in df.astype(object).iterrows():
        lines.append(_tex(idx) + " & " + " & ".join(_cell(v, digits) for v in row) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    if notes:
        lines += [r"\\[0.5em]", r"\begin{minipage}{0.95\textwidth}",
                  r"\scriptsize \textit{Notes.} " + notes, r"\end{minipage}"]
    lines.append(r"\end{table}")
    path = os.path.join(TAB_DIR, fname)
    with open(path, "w") as fh:
        fh.write("\n".join(lines) + "\n")
    print("saved", path)

## Which characteristics drive the forecasts (Section 5.2)

Importance: mean absolute SHAP value over all test observations, as a share of the total. The rank correlation with the gain importance of notebook 08 is a check.

In [ ]:
imp = {(s, m): importance(S[s], m) for s in S for m in TREES}
imp_share = pd.DataFrame({k: shares(v) for k, v in imp.items()})
imp_share = imp_share.loc[imp_share.mean(axis=1).sort_values(ascending=False).index]
print("Share of mean |SHAP| (%):")
display(imp_share.round(1))
print("Mean |SHAP| (percentage points of monthly return):")
display(pd.DataFrame(imp).loc[imp_share.index].round(3))

# Check against the gain importance of notebook 08 (rank correlation of the average shares)
for tag, s in SAMPLES.items():
    for m in TREES:
        f = os.path.join(DATA_DIR, f"importance_{m}_{tag}.csv")
        if os.path.exists(f):
            gain = pd.read_csv(f, index_col=0).mean(axis=1)
            gain = gain.rename({"country (all)": "country", "sector (all)": "sector"})
            both = pd.concat([gain, imp_share[(s, m)]], axis=1).dropna()
            rho = both.rank().corr().iloc[0, 1]
            print(f"{s:14s} {m:5s} rank correlation SHAP vs gain importance: {rho:.2f}")

latex_table(imp_share, "tab_shap_importance.tex",
            "Importance of the characteristics: share of the mean absolute SHAP value (\\%), 2010--2025",
            "tab:shap_importance", digits=1,
            notes="Mean absolute SHAP value of each characteristic over all test observations, as a "
                  "share of the total. Country and sector indicators are grouped.")
imp_share.round(3).to_csv(os.path.join(TAB_DIR, "shap_importance_shares.csv"))

In [ ]:
# Importance by test year (share of mean |SHAP|), for the stability over time
by_year = {}
for s in S:
    for m in TREES:
        g = S[s].groupby(S[s]["date"].dt.year)[shap_cols(m)].apply(lambda d: d.astype("float64").abs().mean())
        g.columns = FEATURES
        by_year[(s, m)] = 100 * g.div(g.sum(axis=1), axis=0)
top = imp_share.index[:4].tolist()
for k, v in by_year.items():
    v.round(2).to_csv(os.path.join(TAB_DIR, f"shap_by_year_{k[1]}_{k[0].replace(' ', '_').lower()}.csv"))
    print(k, "- most important characteristic by year:", v.idxmax(axis=1).value_counts().to_dict())

## Figures

In [ ]:
COLORS = {"rf": "#eb6834", "gbrt": "#1baf7a"}
FEAT_COLORS = ["#008300", "#6250d6", "#e34948", "#2a78d6"]
INK, GRID = "#52514e", "#e5e4df"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK, "axes.labelcolor": INK,
                     "xtick.color": INK, "ytick.color": INK, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})


def style(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)


# Importance of the 12 most important characteristics, rf and gbrt, both samples
top12 = imp_share.index[:12][::-1]
fig, axes = plt.subplots(1, 2, figsize=(6.5, 4.0), sharey=True)
h = 0.38
for j, s in enumerate(S):
    ax = axes[j]
    yy = np.arange(len(top12))
    for k, m in enumerate(TREES):
        ax.barh(yy + (k - 0.5) * h, imp_share.loc[top12, (s, m)], height=h * 0.9,
                color=COLORS[m], label=m)
    ax.set_yticks(yy)
    ax.set_yticklabels(top12)
    ax.set_title(s, fontsize=10, color=INK)
    ax.set_xlabel("Share of mean |SHAP| (%)")
    style(ax, "x")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(os.path.join(FIG_DIR, "fig_shap_importance.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Figure 5.1: importance of the four most important characteristics by test year (rf)
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.9), sharey=True)
for j, s in enumerate(S):
    ax = axes[j]
    v = by_year[(s, "rf")]
    for c, col in zip(top, FEAT_COLORS):
        ax.plot(v.index, v[c], color=col, lw=1.6, marker="o", ms=3, label=c)
    ax.set_title(s, fontsize=10, color=INK)
    ax.set_ylabel("Share of mean |SHAP| (%)" if j == 0 else "")
    ax.set_xticks(range(2010, 2026, 3))
    style(ax)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(labels), loc="upper center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(os.path.join(FIG_DIR, "fig_shap_by_year.pdf"), bbox_inches="tight")
plt.show()

## Nonlinearities and interactions (Section 5.3)

Average SHAP value of the four most important characteristics within 20 equal-width intervals of their cross-sectional rank (−1 to +1); then the same curves for firms above and below the monthly median market value.

In [ ]:
# Figure 5.2: average SHAP value by interval of the cross-sectional rank
BINS = np.linspace(-1, 1, 21)
MID = (BINS[:-1] + BINS[1:]) / 2


def binned_shap(df, char, model):
    b = np.clip(np.digitize(df[char], BINS) - 1, 0, len(MID) - 1)
    m = (100 * df[f"{model}__{char}"]).groupby(b).mean()
    return pd.Series(m.values, index=MID[m.index])


dep_chars = [c for c in imp_share.index if c in CHARS][:4]
fig, axes = plt.subplots(2, 2, figsize=(6.5, 5.0))
for ax, c in zip(axes.ravel(), dep_chars):
    for m in TREES:
        for s, ls in (("Complete case", "--"), ("Imputed", "-")):
            b = binned_shap(S[s], c, m)
            ax.plot(b.index, b.values, color=COLORS[m], ls=ls, lw=1.6, label=f"{m}, {s.lower()}")
    ax.axhline(0, color=INK, lw=0.6)
    ax.set_title(c, fontsize=10, color=INK)
    ax.set_xlabel("Characteristic (cross-sectional rank)")
    ax.set_ylabel("SHAP value (% per month)")
    style(ax)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(os.path.join(FIG_DIR, "fig_shap_dependence.pdf"), bbox_inches="tight")
plt.show()

dep_tab = {}
for c in dep_chars:
    for m in TREES:
        for s in S:
            b = binned_shap(S[s], c, m)
            dep_tab[(c, m, s)] = {"lowest bin": b.iloc[0], "middle": b.iloc[len(b) // 2], "highest bin": b.iloc[-1]}
print("Average SHAP value (% per month) in the lowest, middle and highest bin:")
display(pd.DataFrame(dep_tab).T.round(3))

In [ ]:
# Interaction with firm size, for the two most important characteristics (imputed sample)
s = "Imputed"
df = S[s]
large = df["MV"] >= df.groupby("date")["MV"].transform("median")
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.9))
size_tab = {}
for ax, c in zip(axes, dep_chars[:2]):
    for m in TREES:
        for grp, mask, ls in (("small firms", ~large, "-"), ("large firms", large, "--")):
            b = binned_shap(df[mask], c, m)
            ax.plot(b.index, b.values, color=COLORS[m], ls=ls, lw=1.6, label=f"{m}, {grp}")
            size_tab[(c, m, grp)] = {"lowest bin": b.iloc[0], "highest bin": b.iloc[-1],
                                     "range": b.iloc[-1] - b.iloc[0]}
    ax.axhline(0, color=INK, lw=0.6)
    ax.set_title(c, fontsize=10, color=INK)
    ax.set_xlabel("Characteristic (cross-sectional rank)")
    ax.set_ylabel("SHAP value (% per month)" if ax is axes[0] else "")
    style(ax)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.84))
fig.savefig(os.path.join(FIG_DIR, "fig_shap_size.pdf"), bbox_inches="tight")
plt.show()
print("Imputed sample, average SHAP value (% per month) by firm size:")
display(pd.DataFrame(size_tab).T.round(3))

## Calm and stress months (Section 5.4)

Market return: value-weighted average of the non-winsorised excess returns of the imputed sample. A month is a stress month if the 12-month volatility of the market return, current month included, is above its median over 2010–2025. The classification is ex post and the same for both samples; measures by regime are pooled over the months of each regime.

In [ ]:
# Market return and its 12-month volatility, from the imputed estimation sample (2000-2025)
est_i = load_sample("imputed_mv25", DATA_DIR, DATA_DIR)[["date", "raw_target", "MV"]]
mkt = (est_i["raw_target"] * est_i["MV"]).groupby(est_i["date"]).sum() / est_i.groupby("date")["MV"].sum()
vol = mkt.rolling(12).std()
test_months = vol.index[(vol.index.year >= FIRST_TEST) & (vol.index.year <= LAST_TEST)]
threshold = vol.loc[test_months].median()
stress = (vol.loc[test_months] > threshold).rename("stress")
del est_i; gc.collect()

print(f"Stress months: {int(stress.sum())} of {len(stress)} (volatility above {100 * threshold:.2f}% per month)")
print("Stress months by year:", stress.groupby(stress.index.year).sum().to_dict())
stress.astype(int).to_csv(os.path.join(TAB_DIR, "regimes.csv"))

# Figure 5.3: market volatility and stress months
fig, ax = plt.subplots(figsize=(6.5, 2.6))
v = 100 * vol.loc[test_months]
ax.fill_between(v.index, 0, v.max() * 1.05, where=stress.values, color=GRID, step="mid", lw=0)
ax.plot(v.index, v.values, color=INK, lw=1.4)
ax.axhline(100 * threshold, color=INK, lw=0.8, ls="--")
ax.set_ylabel("12-month volatility of\nmarket return (%)")
ax.set_ylim(0, v.max() * 1.05)
ax.xaxis.set_major_locator(mdates.YearLocator(3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
style(ax)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig_regimes.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# R^2 by regime, from the final forecasts of notebooks 07-09 (same models as Chapter 4)
NN = ["nn_1", "nn_2", "nn_3", "nn_4", "nn_5"]
REG_MODELS = ["ols3", "ols_pars", "elanet", "rf", "gbrt", "nn_3", "comb"]


def nw_tstat(x, lags=4):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    T = len(x)
    u = x - x.mean()
    lrv = u @ u / T
    for l in range(1, lags + 1):
        lrv += 2 * (1 - l / (lags + 1)) * (u[l:] @ u[:-l]) / T
    return x.mean() / np.sqrt(lrv / T)


def r2_pooled(df, m, y="target"):
    return 100 * (1 - ((df[y] - df[m]) ** 2).sum() / (df[y] ** 2).sum())


def r2_cs(df, m, y="target"):
    yy = df[y] - df.groupby("date")[y].transform("mean")
    ff = df[m] - df.groupby("date")[m].transform("mean")
    return 100 * (1 - ((yy - ff) ** 2).sum() / (yy ** 2).sum())


P = {}
for tag, s in SAMPLES.items():
    p = pd.read_parquet(os.path.join(DATA_DIR, f"predictions_final_{tag}.parquet"))
    p["date"] = pd.to_datetime(p["date"])
    p["comb"] = (p["elanet"] + p["rf"] + p["gbrt"] + p[NN].mean(axis=1)) / 4
    p[REG_MODELS] = p[REG_MODELS].astype("float64")
    p["stress"] = p["date"].map(stress)
    assert p["stress"].notna().all()
    P[s] = p

r2_reg, cs_reg, dm_reg = {}, {}, {}
for s, p in P.items():
    for reg, flag in (("Calm", False), ("Stress", True)):
        d = p[p["stress"] == flag]
        r2_reg[(s, reg)] = pd.Series({m: r2_pooled(d, m) for m in REG_MODELS})
        cs_reg[(s, reg)] = pd.Series({m: r2_cs(d, m) for m in REG_MODELS})
        e = pd.DataFrame({m: (d["target"] - d[m]) ** 2 for m in REG_MODELS}).groupby(d["date"]).mean()
        dm_reg[(s, reg)] = pd.Series({m: nw_tstat(e["ols3"] - e[m]) for m in REG_MODELS if m != "ols3"})
r2_reg, cs_reg, dm_reg = pd.DataFrame(r2_reg), pd.DataFrame(cs_reg), pd.DataFrame(dm_reg)
print("Pooled R^2 by regime (%):"); display(r2_reg.round(2))
print("Cross-sectional R^2 by regime (%):"); display(cs_reg.round(2))
print("Diebold-Mariano against ols3 by regime (positive = model more accurate than ols3):"); display(dm_reg.round(2))

latex_table(r2_reg, "tab_regime_r2.tex", "Out-of-sample $R^2$ (\\%) in calm and stress months, 2010--2025",
            "tab:regime_r2", notes="Pooled $R^2$ over the months of each regime. Stress months: 12-month "
            "volatility of the value-weighted market return above its median over 2010--2025.")
latex_table(cs_reg, "tab_regime_r2_cs.tex", "Cross-sectional $R^2$ (\\%) in calm and stress months, 2010--2025",
            "tab:regime_r2_cs")
latex_table(dm_reg, "tab_regime_dm.tex", "Diebold--Mariano tests against \\texttt{ols3} in calm and stress months",
            "tab:regime_dm", notes="Positive values indicate that the model is more accurate than \\texttt{ols3}; "
            "Newey--West standard errors with four lags.")

In [ ]:
# SHAP importance by regime (share of mean |SHAP|), random forest and gradient boosting
reg_imp = {}
for s in S:
    df = S[s]
    flag = df["date"].map(stress)
    for m in TREES:
        for reg, f in (("Calm", False), ("Stress", True)):
            reg_imp[(s, m, reg)] = shares(importance(df[flag == f], m))
reg_imp = pd.DataFrame(reg_imp).loc[imp_share.index]
print("Share of mean |SHAP| by regime (%):")
display(reg_imp.round(1))
reg_imp.round(3).to_csv(os.path.join(TAB_DIR, "shap_importance_regimes.csv"))

rf_reg = reg_imp.xs("rf", axis=1, level=1).iloc[:10]
latex_table(rf_reg, "tab_regime_shap.tex",
            "Importance of the characteristics in calm and stress months, random forest (\\%)",
            "tab:regime_shap", digits=1,
            notes="Share of the mean absolute SHAP value over the test observations of each regime; "
                  "the ten most important characteristics overall.")

## Figures quoted in Chapter 5

Share of zero dividend yields among the test observations (Section 5.3), and the share of `ret_2_12` and the two most important characteristics in each test year (Section 5.2).

In [ ]:
# Share of test observations with zero dividend yield (raw value), to read the dy curve
raw = pd.read_parquet(os.path.join(DATA_DIR, "panel_characteristics.parquet"), columns=["symbol", "date", "dy"])
raw["date"] = pd.to_datetime(raw["date"])
for s, df in S.items():
    m = df[["symbol", "date", "dy"]].merge(raw.rename(columns={"dy": "dy_raw"}), on=["symbol", "date"], how="left")
    zero = m["dy_raw"] == 0
    print(f"{s}: dy = 0 for {zero.mean():.1%} of test observations; "
          f"their rank goes from {m.loc[zero, 'dy'].min():.2f} to {m.loc[zero, 'dy'].max():.2f}")

In [ ]:
# Share of ret_2_12 and the two most important characteristics in each test year
for s in ["complete_case", "imputed"]:
    for m in ["rf", "gbrt"]:
        v = pd.read_csv(os.path.join(TAB_DIR, f"shap_by_year_{m}_{s}.csv"), index_col=0)
        top2 = v.apply(lambda r: r.sort_values(ascending=False).index[:2].tolist(), axis=1)
        out = pd.DataFrame({"ret_2_12": v["ret_2_12"].round(1), "first": top2.str[0],
                            "share_1st": v.max(axis=1).round(1), "second": top2.str[1]})
        print(f"\n{s} - {m}"); print(out.to_string())